In [29]:
import pandas as pd

df_sortimento = pd.read_excel('data/SORTIMENTO PARA FCT ALONG - FEV.xlsx', engine='calamine')
df_sortimento.head()

,LOJA,PRODUTO,COR,TAMANHO,MINIMO,ALVO,DIAS_ESTOQUE,VENDAS,DIAS_VENDA,VELOCIDADE_VENDA,COBERTURA,VEL_VDA_COB,RESSUPRIMENTO
0,10,0241,N0A00S,XP,0,NaN,NaN,NaN,NaN,NaN,90,NaN,SIM
1,10,0241,N0A00S,P,3,NaN,NaN,NaN,NaN,NaN,90,NaN,SIM
2,10,0241,N0A00S,M,4,NaN,NaN,NaN,NaN,NaN,90,NaN,SIM
3,10,0241,N0A00S,G,4,NaN,NaN,NaN,NaN,NaN,90,NaN,SIM
4,10,0241,N0A00S,XG,2,NaN,NaN,NaN,NaN,NaN,90,NaN,SIM


In [31]:
df_sortimento['ARTIGO_COR'] = df_sortimento['PRODUTO']  + df_sortimento['COR']

In [33]:
lista_artigos_unicos = df_sortimento['ARTIGO_COR'].dropna().unique() 
string_artigos = ','.join(lista_artigos_unicos)


lista_filiais_unicas = df_sortimento['LOJA'].dropna().astype(str).unique()
string_filiais = ','.join(lista_filiais_unicas)

In [34]:
print("Tamanho da lista de Artigos: ", len(lista_artigos_unicos))
print("Tamanho da lista de Filiais: ", len(lista_filiais_unicas))

Tamanho da lista de Artigos:  160
Tamanho da lista de Filiais:  455


In [36]:
from google.cloud import bigquery
import warnings
warnings.filterwarnings("ignore", message="Your application has authenticated using end user credentials")



client = bigquery.Client(project='hering-data-analytics-prod')


query = f"""


DECLARE lista_artigos STRING;
DECLARE lista_filial  STRING;

SET lista_artigos = '{string_artigos}';
SET lista_filial = '{string_filiais}';



with cte as (
SELECT 
  cast(c.PontoVda as STRING) as PontoVda,
  c.Sit,
  c.Usuario,
  c.ARTIGO_COR,
  c.Tamanho,
  sum(cast(c.Pecas as integer)) as Pecas
FROM `hering-data-analytics-prod.BI.vendas_sell_in_fato_bi` c
WHERE  c.Sit In ('PENDENTE', 'ATENDIDO', 'SUSPENSO COMERCIAL', 'SUSPENSO COMERCIAL + FINANCEIRO', 'SUSPENSO FINANCEIRO') and  DATE_TRUNC(DATE(DataHora_Criacao_OV, 'America/Sao_Paulo'), MONTH) = DATE_TRUNC(CURRENT_DATE('America/Sao_Paulo'), MONTH) and c.Usuario = 'ABF'
and c.ARTIGO_COR  IN UNNEST(SPLIT(lista_artigos, ','))
and CAST(c.PontoVda AS STRING) IN UNNEST(SPLIT(lista_filial, ','))
group by c.PontoVda, c.Sit, c.Usuario, c.ARTIGO_COR, c.Tamanho
order by c.PontoVda asc
)

select * from cte where PontoVda IS NOT NULL 

"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_CA = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_CA)}")
display(df_CA.head())

Rodando a query...
Total de linhas retornadas: 34236


,PontoVda,Sit,Usuario,ARTIGO_COR,Tamanho,Pecas
0,7,PENDENTE,ABF,0111N0A00S,G,3
1,7,ATENDIDO,ABF,K5AGNMCSI,M,1
2,7,PENDENTE,ABF,0201N1007S,M,7
3,7,PENDENTE,ABF,022BN0A00S,G,5
4,7,ATENDIDO,ABF,H4FAN10SI,P,1


In [38]:
df_CA['PontoVda'].nunique()


427

In [39]:
df_CA['ARTIGO_COR'].nunique()

160

In [40]:
df_CA.to_excel('data/df_CA.xlsx', index=False)

In [92]:
query = f"""

DECLARE lista_artigos STRING;
DECLARE lista_filial  STRING;

SET lista_artigos = '{string_artigos}';
SET lista_filial = '{string_filiais}';




SELECT
    A.FilialID,
    A.ArtigoID ARTIGO_COR,
    A.Tamanho,
    A.Data,
    B.TIPO_FILIAL as CANAL,
    D.PONTO_VENDA_COD AS FILIAL,
    cast(coalesce(SUM(A.Q_Pecas), 0) as INT) AS total_pecas,
    coalesce(SUM(A.Q_Estoque_Total), 0) AS estoque_total
FROM
    `hering-data-analytics-prod.BI.performance_comercial_fato_bi` AS A
LEFT JOIN
    `hering-data-analytics-prod.GOLD.dim_filiais_ro` AS B
    ON A.FilialID = B.LOJA_COD
LEFT JOIN 
    `hering-data-analytics-prod.BI.performance_comercial_dim_filiais` as D on A.FilialID = D.LOJA_ID
WHERE
 A.Data = CURRENT_DATE() - 1
    AND B.TIPO_FILIAL = 'FRANQUIAS' 
    AND A.ArtigoID IN UNNEST(SPLIT(lista_artigos, ','))
    AND D.PONTO_VENDA_COD IN UNNEST(SPLIT(lista_filial, ','))
GROUP BY
    A.FilialID,
    A.ArtigoID,
    A.Tamanho,
    A.Data,
    A.Piramide,
    B.TIPO_FILIAL,
    B.PONTO_VENDA,
    D.PONTO_VENDA_COD
ORDER BY
    A.FilialID ASC


"""

# 4. Executa e joga direto para um DataFrame do Pandas
print("Rodando a query...")
df_stk  = client.query(query).to_dataframe()

# 5. Visualiza o resultado
print(f"Total de linhas retornadas: {len(df_stk)}")
display(df_stk.head())

Rodando a query...
Total de linhas retornadas: 316303


,FilialID,ARTIGO_COR,Tamanho,Data,CANAL,FILIAL,total_pecas,estoque_total
0,20,036H8GEN,G,2026-01-28,FRANQUIAS,104,0,4
1,20,4FBZN0AEN,M,2026-01-28,FRANQUIAS,104,0,2
2,20,K48RN0ASI,XXG,2026-01-28,FRANQUIAS,104,0,2
3,20,N1TEN1007S,XXG,2026-01-28,FRANQUIAS,104,0,2
4,20,N3A7N0A00S,G,2026-01-28,FRANQUIAS,104,1,0


In [93]:
df_stk.shape

(316303, 8)

In [94]:
df_stk['estoque_total'].sum()


np.int64(1251810)

In [95]:
df_stk['Estoque Total'] = (df_stk['estoque_total'] - df_stk['total_pecas']).abs()

In [96]:
df_stk.head()

,FilialID,ARTIGO_COR,Tamanho,Data,CANAL,FILIAL,total_pecas,estoque_total,Estoque Total
0,20,036H8GEN,G,2026-01-28,FRANQUIAS,104,0,4,4
1,20,4FBZN0AEN,M,2026-01-28,FRANQUIAS,104,0,2,2
2,20,K48RN0ASI,XXG,2026-01-28,FRANQUIAS,104,0,2,2
3,20,N1TEN1007S,XXG,2026-01-28,FRANQUIAS,104,0,2,2
4,20,N3A7N0A00S,G,2026-01-28,FRANQUIAS,104,1,0,1


In [97]:
df_agregado = df_stk.groupby('Data')[['Estoque Total']].sum().reset_index()
display(df_agregado)

,Data,Estoque Total
0,2026-01-28,1267549


In [98]:
df_agregado = df_stk.groupby('Data')[['total_pecas']].sum().reset_index()
display(df_agregado)

,Data,total_pecas
0,2026-01-28,7689


In [99]:
df_stk.isnull().sum()

FilialID         0
ARTIGO_COR       0
Tamanho          0
Data             0
CANAL            0
FILIAL           0
total_pecas      0
estoque_total    0
Estoque Total    0
dtype: int64

In [100]:
df_stk['Tamanho'].unique()

array(['G', 'M', 'XXG', 'G  ', '038', 'P', 'XG', 'EXG', '040', '036',
       '050', '046', '048', '042', '044', 'P  ', 'XP', '034', '052',
       'M  ', 'XG ', 'G       ', '012', 'XXG     ', '006', '004', '016',
       'M       ', '014', '008', '010', '002', '001', '005', 'XG      ',
       'XP ', 'P       ', '003', 'G1', '056', '054', 'G3'], dtype=object)

In [101]:
df_stk['ARTIGO_COR'].nunique()

160

In [104]:
df_stk.to_excel('data/df_STK.xlsx', index=False)